Coding Question: Image Classification Neural Network

Using the dataset provided below, design and implement a Neural Network for multi-class image classification.

Dataset: Nepali Numbers and Letters Dataset

Your implementation should include:

Data Preparation

Load, preprocess, resize, normalize, and split the dataset into train/validation/test sets.
Neural Network

Hyperparameter Tuning

Evaluation Evaluate the final model using:

Accuracy
Precision
Recall
F1-score
Macro/Weighted F1
Confusion Matrix
Performance Analysis

Deliverable: Complete, reproducible code with a final report comparing the experimented configurations and their performance

In [2]:
import random
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image

import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.22.0-rc0


Set random seeds

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

Inspect the dataset first

In [4]:
DATA_DIR = Path("images")
ARCHIVE_PATH = Path("images.rar")

print("Dataset directory exists:", DATA_DIR.is_dir())
print("Dataset archive exists:", ARCHIVE_PATH.is_file())

Dataset directory exists: False
Dataset archive exists: False


Discover class folders

In [ ]:
if DATA_DIR.is_dir():
    class_dirs = sorted(
        [path for path in DATA_DIR.iterdir() if path.is_dir()],
        key=lambda path: (not path.name.isdigit(), path.name),
    )
    class_names = [path.name for path in class_dirs]
else:
    files = subprocess.check_output(
        ["bsdtar", "-tf", str(ARCHIVE_PATH)], text=True
    ).splitlines()
    class_names = sorted({
        path.split("/")[1]
        for path in files
        if path.startswith("images/") and len(path.split("/")) > 2
    }, key=lambda name: (not name.isdigit(), name))
    class_dirs = []

print("Number of classes:", len(class_names))
for i, name in enumerate(class_names):
    print(i, "->", name)

In [ ]:
# Count images in every class

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

class_counts = {
    name: sum(
        file.suffix.lower() in image_extensions
        for file in (DATA_DIR / name).rglob("*")
    )
    for name in class_names
}

class_counts_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["class", "image_count"],
)

class_counts_df

In [ ]:
plt.figure(figsize=(15, 6))

sns.barplot(
    data=class_counts_df,
    x="class",
    y="image_count"
)

plt.xticks(rotation=90)
plt.title("Number of Images per Class")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

In [ ]:
sample_files = [
    next(
        file for file in (DATA_DIR / name).rglob("*")
        if file.suffix.lower() in image_extensions
    )
    for name in class_names
]

fig, axes = plt.subplots(2, 6, figsize=(15, 5))

for axis, file in zip(axes.flat, sample_files):
    axis.imshow(Image.open(file))
    axis.set_title(file.parent.name)
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Create Image Dataframe

image_paths = []
labels = []

class_to_index = {
    name: index
    for index, name in enumerate(class_names)
}

for class_name in class_names:
    for image_path in (DATA_DIR / class_name).rglob("*"):
        if image_path.suffix.lower() in image_extensions:
            image_paths.append(str(image_path))
            labels.append(class_to_index[class_name])

df = pd.DataFrame({
    "image_path": image_paths,
    "label": labels,
})
df["class_name"] = df["label"].map(
    {index: name for name, index in class_to_index.items()}
)

print("Total images:", len(df))
df.head()

In [ ]:
# Check for corrupted images
def check_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


df["valid"] = df["image_path"].apply(check_image)

print(df["valid"].value_counts())

In [ ]:
if "valid" in df.columns:
    valid_mask = df["valid"].to_numpy(dtype=bool)
    df = df.loc[valid_mask].drop(columns="valid", errors="ignore")

print("Valid images:", len(df))

Prepare train, validation, and test data

In [ ]:
IMAGE_SIZE = (32, 32)

if DATA_DIR is None or not DATA_DIR.is_dir():
    raise FileNotFoundError("Extract images.rar into an images/ folder before training.")

image_paths = df["image_path"].to_numpy()
labels = df["label"].to_numpy()

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels,
)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

def load_images(paths):
    images = []
    for path in paths:
        with Image.open(path) as image:
            images.append(np.asarray(image.convert("RGB").resize(IMAGE_SIZE), dtype="float32") / 255.0)
    return np.stack(images)

X_train = load_images(X_train_paths)
X_val = load_images(X_val_paths)
X_test = load_images(X_test_paths)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
# Neural network and hyperparameter experiments

In [ ]:
def make_model(filters=32, dropout=0.3):
    model = Sequential([
        tf.keras.Input((*IMAGE_SIZE, 3)),
        Conv2D(filters, 3, activation="relu"),
        MaxPooling2D(),
        Conv2D(filters * 2, 3, activation="relu"),
        MaxPooling2D(),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(dropout),
        Dense(len(class_names), activation="softmax"),
    ])
    model.compile(
        optimizer=Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

results = []
for filters in [16, 32]:
    model = make_model(filters)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=32,
        verbose=0,
    )
    results.append({
        "filters": filters,
        "val_accuracy": max(history.history["val_accuracy"]),
    })

experiment_df = pd.DataFrame(results).sort_values(
    "val_accuracy", ascending=False
)
experiment_df

In [ ]:
# Train and evaluate the final model

best_filters = int(experiment_df.iloc[0]["filters"])
final_model = make_model(best_filters, dropout=0.4)

final_history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True
    )],
    verbose=1,
)

y_pred = final_model.predict(X_test, verbose=0).argmax(axis=1)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

print(pd.Series(metrics).round(4))

In [ ]:
best_filters = int(experiment_df.iloc[0]["filters"])
final_model = make_model(best_filters, dropout=0.4)

final_history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True
    )],
    verbose=1,
)

y_pred = final_model.predict(X_test, verbose=0).argmax(axis=1)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

print(pd.Series(metrics).round(4))

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(final_history.history["accuracy"], label="train")
plt.plot(final_history.history["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(final_history.history["loss"], label="train")
plt.plot(final_history.history["val_loss"], label="validation")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()

print("Best filters:", best_filters)
print("Test accuracy:", round(metrics["accuracy"], 4))
print("Macro F1:", round(metrics["macro_f1"], 4))
print("Weighted F1:", round(metrics["weighted_f1"], 4))

Performance analysis

The selected CNN uses the best filter size from the validation experiment. The final test metrics and confusion matrix show overall and per-class performance. The training curves help identify overfitting. More augmentation, regularization, or additional tuning could improve weaker classes.